# Resume2Vec — Training & Evaluation Notebook
This notebook walks through every stage of the pipeline:
1. Synthetic data generation
2. Siamese model training
3. T5 optimizer training
4. Evaluation metrics
5. Demo inference

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from loguru import logger
print('Setup OK')

## 1. Generate Synthetic Training Data

In [ ]:
from src.generate_synthetic_data import generate
df_siamese, df_opt = generate(n_per_domain=200)  # small for notebook speed
print(df_siamese.head())
print(f'\nClass balance:\n{df_siamese["label"].value_counts()}')

## 2. Visualise keyword distributions

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import re

all_text = ' '.join(df_siamese['jd_text'].tolist())
tokens   = re.findall(r'\b[A-Za-z][A-Za-z0-9+#.-]{2,}\b', all_text)
freq     = Counter(tokens).most_common(20)
words, counts = zip(*freq)

plt.figure(figsize=(12, 4))
plt.bar(words, counts, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 Keywords in Job Descriptions')
plt.tight_layout()
plt.show()

## 3. Train Siamese Model (mini, 2 epochs)

In [ ]:
from src.siamese_model import train as train_siamese
train_siamese(csv_path='../data/processed/pairs.csv', epochs=2)

## 4. Evaluate Siamese Model

In [ ]:
from evaluate import evaluate_siamese
results = evaluate_siamese(n_samples=200)
print(results)

## 5. Demo: Score a Resume Against a Job Description

In [ ]:
from src.siamese_model import load_model, get_match_score

model, tokenizer = load_model('../models/siamese_checkpoint.pt')

resume = '''
Jane Doe — Python Developer
Skills: Python, Django, PostgreSQL, Docker, Git
Experience: 3 years at TechCorp building REST APIs.
Education: BSc Software Engineering.
'''

jd = '''
We are looking for a Python Developer with Django and Docker skills.
You will build and maintain REST APIs and work with PostgreSQL databases.
'''

score = get_match_score(resume, jd, model=model, tokenizer=tokenizer)
print(f'Semantic Match Score: {score:.4f}  ({"✅ Good match" if score > 0.5 else "❌ Low match"})')

## 6. Demo: Full ATS Report

In [ ]:
from src.ats_checker import ats_score
import json

report = ats_score(resume, jd)
print(f'Overall ATS Score: {report["overall_score"]} / 100')
print(json.dumps(report['breakdown'], indent=2))
print('Missing keywords:', report['keyword_coverage']['missing'])

## 7. Demo: Optimize Resume with GenAI

In [ ]:
from src.resume_optimizer import ResumeOptimizer

optimizer = ResumeOptimizer()   # loads T5 if trained, else distilGPT-2 fallback
optimized = optimizer.optimize(resume, jd)
print('--- Optimized Resume ---')
print(optimized)

new_report = ats_score(optimized, jd)
print(f'\nATS Before: {report["overall_score"]}  →  After: {new_report["overall_score"]}')